# Даалгавар 2: Data Scrapping

## Task:
Өөрийн сонгон авсан системээс өгөгдлүүдийг гарган авах data scrapping code бичнэ. Data scrap хийхдээ BeautifulSoup and Google colab ашиглана.

Бүх датаг цуглуулаад өгөгдөлөө tsv өргөтгөлтэй хадгалаад явуулна. Гарган авч буй өгөгдөлд тавигдах гол шаардлага нь дор хаяж 7-н баганатай байхыг шаардана. Үүнд линк болон unique identifier орохгүй.  Хэрвээ өгөгдөл нь том хэмжээтэй бол Google drive руу хуулаад линкээ явуулна.

## 1. Сонгон авсан систем: Arxiv
[Arxiv](https://arxiv.org) системээс гараас оноосон түлхүүр үгийн дагуу буцаасан эрдэм шинжилгээний өгүүллэгүүдийн мэдээллийг хуулах. Хуулсан мэдээлэлд дараах орно:
1. Arxiv code
2. Гарчиг
3. Зохиолчид
4. Тойм
5. Хэвлэгдсэн сэтгүүл
6. Илгээсэн огноо
7. Зарлагдсан огноо

## 2. Google Scholar
[Google Scholar](https://scholar.google.com/) системээс 1-р алхамд хуулсан бүх зохиолчдын мэдээллийг хуулах. Эдгээр нь:
1. Судалгааны чиглэл
2. Сургууль

## 3. Co-authorship сүлжээ байгуулах
Гараас оноосон түлхүүр үгийн хайлтын илэрцээс олдсон судлаачдын сүлжээг байгуулах. (just for fun)

In [60]:
import os
import re
import time
import pickle
import psutil
from tqdm import tqdm
from unicodedata import normalize
from pprint import pprint

import igraph as ig
import networkx as nx
import chart_studio.plotly
from chart_studio.plotly import plot, iplot

import multiprocessing
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

from jellyfish import jaro_winkler_similarity

import seaborn as sns
import plotly.graph_objs as go
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

pool = ThreadPoolExecutor()
default_workers_threads = pool._max_workers

print(f"CPU count: {os.cpu_count()}")
print(f"Memory GB: {psutil.virtual_memory().total >> 30}")
print(f"Default thread workers: {default_workers_threads}")

CPU count: 16
Memory GB: 30
Default thread workers: 20


## 1. Arxiv

In [2]:
num_workers = 16

# Arxiv нэг хуудсанд гарах өгүүллэлийг тоог 200 гэж заасан тул хуудас хооронд 200-р шилжинэ (increment)
start_page = 0
end_page = 5800
increment = 200

# Advanced search дээр хайлтын түлхүүр үгээ оруулна. Энэ тохиолдолд 'audio recognition'
search_term = 'audio+recognition'
base_link = f"https://arxiv.org/search/advanced?advanced=&terms-0-operator=AND&terms-0-term={search_term}&terms-0-field=all&classification-physics_archives=all&classification-include_cross_list=include&date-filter_by=all_dates&date-year=&date-from_date=&date-to_date=&date-date_type=submitted_date&abstracts=show&size=200&order=-announced_date_first"

papers = []

# Хуудас бүрийн мэдээллийг авахад бид олон удаа request үүсгэж байгаа. Иймд нэг удаа үүсгэсэн connection-г олон дахин ашиглах үүднээс 
# session тодорхойлно
session = requests.Session()

def scrape_papers(start_num):
    linkie = f"{base_link}&start={start_num}"

    html_text = session.get(linkie).text
    soup = BeautifulSoup(html_text, 'html.parser')

    papers_sub = soup.find_all('li', class_ = 'arxiv-result')

    papers.extend(papers_sub)

In [3]:
# 2-р хэсэгт зохиолч бүрийн мэдээллийг нэг нэгээр хуулах нь дор хаяж 1 цаг болж байсан. Иймд эхлээд 2-р хэсэгт параллель тооцоо ашигласан ба энэ хэсэгт ердөө
# хийсэн зүйлээ бататгах зорилгоор ашигласан юм.
with ThreadPoolExecutor(max_workers = num_workers) as executor, tqdm(total = (end_page - start_page), 
                                                                     desc = f'Scraping papers from arxiv, [SEARCH TERM]: {search_term}') as pbar:
    futures = [executor.submit(scrape_papers, start_num) for start_num in range(start_page, end_page, increment)]
    for future in concurrent.futures.as_completed(futures):
        pbar.update(increment)

print(f"Number of papers scraped: {len(papers)}")

Scraping papers from arxiv, [SEARCH TERM]: audio+recognition: 100%|██████████| 5800/5800 [00:22<00:00, 257.68it/s]

Number of papers scraped: 3200


In [4]:
def punct(full_text):
    temp = [sent.strip() for sent in re.findall("""\s+[^.!?]*[.!?]""", full_text)]
    temp = re.sub('\..', '.', '. '.join(temp))
    temp = re.sub('\s+[a-zA-Z]\.', '', temp)
    
    return temp

In [5]:
arxiv_code = []
titles = []
authors = []
abstracts = []
journal = []
submitted_dates = []
originally_announced_dates = []

patterns = {
    'submitted to': r'submitted to (.+)',
    'Accepted to': r'Accepted to (.+)',
    'Accepted in': r'Accepted in (.+)',
    'accepted by': r'accepted by (.+)',
    'Journal ref': r'Journal ref: (.+)'
}

for paper in tqdm(papers, desc = 'Scraping fields'):
    # Arxiv code
    paper_arxiv_code = paper.find('p', class_ = 'list-title is-inline-block').text.split('\n')[0]

    # Paper Title
    paper_title = paper.find('p', class_ = 'title is-5 mathjax').text.strip()
    
    # Authors
    paper_authors = paper.find('p', class_ = 'authors')
    paper_authors = ','.join([name.text.strip() for name in paper_authors.find_all('a')])

    # Abstract
    paper_abstract = paper.find('p', class_ = 'abstract mathjax')
    paper_abstract = punct(paper_abstract.find('span', class_ = 'abstract-full has-text-grey-dark mathjax').text)

    # Comment, which includes submitted Journals etc.
    # Аливаа өгүүллэгийн хувьд хэвлэгдсэн сэтгүүлийн мэдээлэл нь цөөн тооны pattern-ийн дагуу бичигдсэн байсан.
    # Иймд энэ хэсэгт pattern бүрийг шалгана.
    paper_comment = paper.find('p', class_='comments is-size-7')
    paper_journal = np.nan
    if paper_comment:
        paper_comment_text = re.sub(r'\s+', ' ', paper_comment.text.strip())
        for keyword, pattern in patterns.items():
            if keyword in paper_comment_text:
                match = re.search(pattern, paper_comment_text)
                if match:
                    paper_journal = match.group(1).strip()
                    break
    # Submission date
    # Сүүлд нь бүх хугацааг харсан. Тэгэхэд хугацаа бүр ижил форматтай байсан.
    # Иймд ганц удаа split ашиглаж илгээсэн огноог гаргах боломжтой байсныг ойлгосон. Гэхдээ анх ялгаатай pattern-тай огноо байх вий хэмээн болгоомжилж
    # бичсэн аргаа үлдээсэн.
    submit_date = paper.find('p', class_ = 'is-size-7').text.split(';')[0]
    submit_date = re.findall('([0-9]{1,2}\s(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|(Nov|Dec)(?:ember)?)\, [0-9]{4})', submit_date)
    submit_date = [sent for sent in submit_date[0] if len(sent)][0]

    originally_announced_date = paper.find('p', class_ = 'is-size-7').text.strip().split('originally announced ')[-1][:-1]
    
    arxiv_code.append(paper_arxiv_code)
    titles.append(paper_title)
    authors.append(paper_authors)
    abstracts.append(paper_abstract)
    journal.append(paper_journal)
    submitted_dates.append(submit_date)
    originally_announced_dates.append(originally_announced_date)

Scraping fields: 100%|██████████| 3200/3200 [00:02<00:00, 1282.47it/s]


In [6]:
df = pd.DataFrame({'title': titles, 
                   'authors': authors,
                   'abstract': abstracts,
                   'Journal': journal,
                   'code': arxiv_code,
                   'submitted_date': submitted_dates,
                   'orginally_announced_date': originally_announced_dates})
df['submitted_date'] = pd.to_datetime(df['submitted_date'])
df['orginally_announced_date'] = pd.to_datetime(df['orginally_announced_date'], errors = 'coerce')

df.head()

,title,authors,abstract,Journal,code,submitted_date,orginally_announced_date
0,Speech Emotion Recognition Via CNN-Transforemr...,"Xiaoyu Tang,Yixin Lin,Ting Dang,Yuanfang Zhang...",Speech Emotion Recognition (SER) is crucial in...,NaN,arXiv:2403.04743,2024-03-07,2024-03-01
1,Dynamic Cross Attention for Audio-Visual Perso...,"R. Gnana Praveen,Jahangir Alam",Although person or identity verification has b...,FG2024,arXiv:2403.04661,2024-03-07,2024-03-01
2,Audio-Visual Person Verification based on Recu...,"R. Gnana Praveen,Jahangir Alam",Person or identity verification has been recen...,FG2024,arXiv:2403.04654,2024-03-07,2024-03-01
3,CAT: Enhancing Multimodal Large Language Model...,"Qilang Ye,Zitong Yu,Rui Shao,Xinyu Xie,Philip ...",This paper focuses on the challenge of answeri...,NaN,arXiv:2403.04640,2024-03-07,2024-03-01
4,A New Benchmark for Evaluating Automatic Speec...,"Qusai Abo Obaidah,Muhy Eddin Zater,Adnan Jalju...",This work is an attempt to introduce a compreh...,NaN,arXiv:2403.04280,2024-03-07,2024-03-01


In [7]:
df.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     3200 non-null   object        
 1   authors                   3200 non-null   object        
 2   abstract                  3200 non-null   object        
 3   Journal                   474 non-null    object        
 4   code                      3200 non-null   object        
 5   submitted_date            3200 non-null   datetime64[ns]
 6   orginally_announced_date  3200 non-null   datetime64[ns]
dtypes: datetime64[ns](2), object(5)
memory usage: 5.0 MB


In [8]:
output_filename = f"arxiv_{search_term}_search_term"
df.to_parquet(f"{output_filename}.parquet")

In [9]:
df.to_csv(f"{output_filename}.tsv", sep = "\t")

## 2. Google Scholar

In [10]:
# Өгүүлэл бүрийн хувьд хамт ажилласан судлаачдын хүснэгтийг байгуулах.
author_pairs = []

for authors in df['authors']:
    author_list = authors.split(',')
    author_pairs.extend([(author.strip(), other_author.strip()) for i, author in enumerate(author_list) for other_author in author_list[i+1:]])

author_pair_df = pd.DataFrame(author_pairs, columns = ['Author1', 'Author2'])

print(f"Authors network shape: {author_pair_df.shape}")
author_pair_df.head()

Authors network shape: (48788, 2)


,Author1,Author2
0,Xiaoyu Tang,Yixin Lin
1,Xiaoyu Tang,Ting Dang
2,Xiaoyu Tang,Yuanfang Zhang
3,Xiaoyu Tang,Jintao Cheng
4,Yixin Lin,Ting Dang


### Судлаачдын нэрийг зөв олох

Энэ хэсэгт судлаачдын мэдээллийг үнэн зөв хуулахын тулд судлаачдын нэрийг үнэн зөв тааруулах буюу олох шаардлагатай байсан. Иймд google scholar дээрээс судлаачдын нэрийг тааруулахад **Jaro-Winkler Зай** тооцсон ба гараас босго тогтоосон.

Энэ зай нь гараас оноосон $s_1$, $s_2$ тэмдэгт мөрийн хувьд хоорондоо таарсан тэмдэгтийн жинлэсэн давтамжийг тооцдог. 

$$sim_j = \begin{cases}
0 & \text{if } m = 0 \\
\frac{1}{3} \left[ \frac{m}{|s_1|} + \frac{m}{|s_2|} + \frac{m-t}{m}\right] & \text{otherwise}
\end{cases}$$

энд:

* $|s_i|$ тэмдэгт мөрийн урт
* $m$ хоорондоо таарсан тэмдэгтийн тоо
* $t$ нийт удаа хийгдсэн шилжилтийн тоо

<ins>Тодорхойлолт:</ins> $s_1$, $s_2$-н 2 тэмдэгтүүд нь хоорондоо ижил байх нөхцөл нь:
1. Ижил тэмдэгт
2. Хоорондоо $\floor{\frac{max(|s_1|, |s_2|)}{2}}$ тэмдэгтээс холгүй зайтай байх

Хоорондоо таарсан тэмдэгтийн жишээ авъя. `FAREMVIEL` болон `FARMVILLE`. **F**, **A**, **R** ижил байрлалд байна. Харин `M`, `V`, `I`, `E`, `L` 3 тэмдэгтийн зайд байна. Иймд нийт 8 тэмдэгт хоорондоо таарч байна.

Шилжилтийн тоо гэдэг нь $s_1$, $s_2$ мөрүүдийн хоорондоо таарч байгаа гэхдээ ялгаатай байрлалд байгаа тэмдэгтүүдийн тоог 2-т хуваасан тоо юм. Дээрх мөрүүдийн хувьд ялгаатай байрлалд байгаа тэмдэгт нь `E`, `L`.

Иймд `FAREMVIEL` болон `FARMVILLE` хоорондох зай (төсөө) нь $\frac{1}{3} (\frac{8}{9} + \frac{8}{9} + \frac{8-1}{8}) = 0.88$

Source: [Jaro–Winkler distance](https://en.wikipedia.org/wiki/Jaro%E2%80%93Winkler_distance)

In [73]:
3347*5/(4*3600)

1.1621527777777778

In [61]:
session = requests.Session()

# Unique нэр бүрийн хувьд google scholar дээрээс уг нэрийг оруулан хайлт хийнэ.
# Хайлтаас илэрсэн нэр бүртэй төсөөг нь тооцно. Гараас оруулсан босгыг хангаж байвал нэг хүн гэж үзнэ.
def return_matching_profile_link(base_link, author, cut_off = 0.6):
    author_search_name = author.replace(' ', '+')
    linkie = f'https://scholar.google.com/citations?hl=en&view_op=search_authors&mauthors={author_search_name}&btnG='

    time.sleep(3)

    html_text = session.get(linkie).text
    soup = BeautifulSoup(html_text, 'html.parser')

    profile = soup.find_all('div', class_ = 'gsc_1usr')

    jaro_similarity_scores = []
    hit_names = []
    links = []

    if profile:
        for hit in profile:
            hit = hit.find('h3', class_='gs_ai_name').find('a')
            hit_name = hit.text.strip()
            hit_link = base_link + hit['href']

            # Jaro-Winkler төсөөг тооцох.
            jaro_similarity_score = jaro_winkler_similarity(author, hit_name)

            if jaro_similarity_score >= cut_off:
                jaro_similarity_scores.append(jaro_similarity_score)
                links.append(hit_link)
                hit_names.append(hit_name)

    if len(jaro_similarity_scores):
        # Нэгээс олон нэр босго хангах боломжтой учир тэдгээрээс хамгийн өндөр оноотойг сонгох.
        max_score_ind = np.argmax(jaro_similarity_scores)
        max_score_profile_name = hit_names[max_score_ind]
        profile_link = links[max_score_ind]
    else:
        profile_link = None
        max_score_profile_name = np.nan

    return profile_link, max_score_profile_name

In [62]:
# Судлаачийг google scholar-с зөв олсон гэж үзэн холбогдох мэдээллүүдийг хуулах.
def return_author_info(profile_link):
    if profile_link:
        user_html_text = session.get(profile_link).text
        user_soup = BeautifulSoup(user_html_text, 'html.parser')

        institute = user_soup.find('div', class_='gsc_prf_il')
        research_area = user_soup.find('div', {"id": "gsc_prf_int"}, class_='gsc_prf_il')

        if institute:
            institute = institute.text.strip()
        if research_area:
            research_area = [item.text.lower().strip() for item in research_area.find_all('a')]
        elif (not institute) and (not research_area):
            institute = np.nan
            research_area = np.nan
    else:
        institute = np.nan
        research_area = np.nan

    return institute, research_area

In [27]:
hit_names = []
author_institute = []
author_research_area = []
author_unique_names = pd.concat([author_pair_df['Author1'], author_pair_df['Author2']]).unique().tolist()

base_link = 'https://scholar.google.com'

def process_author(name):
    profile_link, min_score_profile_name = return_matching_profile_link(base_link, name)
    institute, research_area = return_author_info(profile_link)

    if isinstance(research_area, list):
        research_area = ','.join(research_area)

    hit_names.append(min_score_profile_name)
    author_institute.append(institute)
    author_research_area.append(research_area)

Нийт нэрийн тооноос хамааран программ 2-5 цаг ажиллах тохиолдлууд гарч байсан. Манай Математик Хэрэглээний Төвийн компьютерын үзүүлэлт дажгүй тул судлаачийн мэдээллийг хуулах ажлыг салаалах буюу параллелиар хийхээр шийдсэн.

In [28]:
with ThreadPoolExecutor(max_workers = num_workers) as executor:
    future_to_author = {executor.submit(process_author, name): name for name in author_unique_names}

    for future in tqdm(concurrent.futures.as_completed(future_to_author), 
                       total = len(future_to_author), desc = "Processing Authors"):
        author_name = future_to_author[future]
        try:
            future.result()
        except Exception as e:
            print(f"Error processing author {author_name}: {e}")

Processing Authors: 100%|██████████| 9084/9084 [05:07<00:00, 29.57it/s]


In [65]:
session = requests.Session()

author = 'Xiaoyu Tang'
author_search_name = author.replace(' ', '+')
linkie = f'https://scholar.google.com/citations?hl=en&view_op=search_authors&mauthors={author_search_name}&btnG='

html_text = session.get(linkie).text
soup = BeautifulSoup(html_text, 'html.parser')

profile = soup.find_all('div', class_ = 'gsc_1usr')

jaro_similarity_scores = []
hit_names = []
links = []

In [52]:
soup.find_all('div', class_ = 'gs_ai gs_scl gs_ai_chpr')

[]

In [57]:
soup.select('div.gsc_1usr')

[]

In [66]:
soup

<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN">

<html>
<head><meta content="text/html; charset=utf-8" http-equiv="content-type"/><meta content="initial-scale=1" name="viewport"/><title>https://scholar.google.com/citations?hl=en&amp;view_op=search_authors&amp;mauthors=Xiaoyu+Tang&amp;btnG=</title></head>
<body onload="e=document.getElementById('captcha');if(e){e.focus();} if(solveSimpleChallenge) {solveSimpleChallenge(,);}" style="font-family: arial, sans-serif; background-color: #fff; color: #000; padding:20px; font-size:18px; overscroll-behavior:contain;">
<div style="max-width:400px;">
<hr noshade="" size="1" style="color:#ccc; background-color:#ccc;"/><br/>
<form action="index" id="captcha-form" method="post">
<noscript>
<div style="font-size:13px;">
  In order to continue, please enable javascript on your web browser.
</div>
</noscript>
<script async="" defer="" src="https://www.google.com/recaptcha/api.js"></script>
<script>var submitCallback = function(response) 

In [43]:
profile

[]

In [39]:
if profile:
    for hit in profile:
        hit = hit.find('h3', class_='gs_ai_name').find('a')
        hit_name = hit.text.strip()
        hit_link = base_link + hit['href']

        # Jaro-Winkler төсөөг тооцох.
        jaro_similarity_score = jaro_winkler_similarity(author, hit_name)

        if jaro_similarity_score >= cut_off:
            jaro_similarity_scores.append(jaro_similarity_score)
            links.append(hit_link)
            hit_names.append(hit_name)

if len(jaro_similarity_scores):
    # Нэгээс олон нэр босго хангах боломжтой учир тэдгээрээс хамгийн өндөр оноотойг сонгох.
    max_score_ind = np.argmax(jaro_similarity_scores)
    max_score_profile_name = hit_names[max_score_ind]
    profile_link = links[max_score_ind]
else:
    profile_link = None
    max_score_profile_name = np.nan

['Xiaoyu Tang',
 'Yixin Lin',
 'Ting Dang',
 'Yuanfang Zhang',
 'R. Gnana Praveen',
 'Qilang Ye',
 'Zitong Yu',
 'Rui Shao',
 'Xinyu Xie',
 'Philip Torr',
 'Qusai Abo Obaidah',
 'Muhy Eddin Zater',
 'Adnan Jaljuli',
 'Ali Mahboub',
 'Asma Hakouz',
 'Bashar Alfrou',
 'Yusheng Dai',
 'Hang Chen',
 'Jun Du',
 'Ruoyu Wang',
 'Shihao Chen',
 'Jiefeng Ma',
 'Haotian Wang',
 'Pedro Ramoneda',
 'Minhee Lee',
 'Dasaem Jeong',
 'J. J. Valero-Mas',
 'Dang Thoai Phan',
 'Andre Jakob',
 'Jorge Álvarez',
 'Juan Carlos Armenteros',
 'Camilo Torrón',
 'Miguel Ortega-Martín',
 'Alfonso Ardoiz',
 'Óscar García',
 'Ignacio Arranz',
 'Íñigo Galdeano',
 'Ignacio Garrido',
 'Adrián Alonso',
 'Fernando Bayón',
 'Tirza Biron',
 'Moshe Barboy',
 'Eran Ben-Artzy',
 'Alona Golubchik',
 'Yanir Marmor',
 'Smadar Szekely',
 'Yaron Winter',
 'Yuxin Guo',
 'Shijie Ma',
 'Hu Su',
 'Zhiqing Wang',
 'Yuhao Zhao',
 'Wei Zou',
 'Siyang Sun',
 'Junwen He',
 'Yifan Wang',
 'Lijun Wang',
 'Huchuan Lu',
 'Jun-Yan He',
 'Jin-P

In [29]:
# Хуулсан мэдээллүүдээ pickle формат болон хүснэгтээс хадгалах
with open('author_institute.pickle', 'wb') as handle:
    pickle.dump(author_institute, handle, protocol = pickle.HIGHEST_PROTOCOL)
    print("saved to author_institute.pickle")

saved to author_institute.pickle


In [30]:
with open('author_research_area.pickle', 'wb') as handle:
    pickle.dump(author_research_area, handle, protocol = pickle.HIGHEST_PROTOCOL)
    print("saved to author_research_area.pickle")

saved to author_research_area.pickle


In [31]:
authors_info = pd.DataFrame({'author': author_unique_names,
                             'hit_name': hit_names,
                             'institute': author_institute,
                             'research_area': author_research_area})
print(f"Authors info df shape: {authors_info.shape}")
authors_info

Authors info df shape: (9084, 4)


,author,hit_name,institute,research_area
0,Xiaoyu Tang,NaN,NaN,NaN
1,Yixin Lin,NaN,NaN,NaN
2,Ting Dang,NaN,NaN,NaN
3,Yuanfang Zhang,NaN,NaN,NaN
4,R. Gnana Praveen,NaN,NaN,NaN
...,...,...,...,...
9079,Ryad Benosman,NaN,NaN,NaN
9080,Giuseppe Riccardi,NaN,NaN,NaN
9081,Jong Kim,NaN,NaN,NaN
9082,Dilek Hakkani-Tür,NaN,NaN,NaN


In [36]:
authors_info.dropna(subset = ['hit_name', 'institute', 'research_area'], how = 'all')

,author,hit_name,institute,research_area


In [32]:
authors_info.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9084 entries, 0 to 9083
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   author         9084 non-null   object 
 1   hit_name       0 non-null      float64
 2   institute      0 non-null      float64
 3   research_area  0 non-null      float64
dtypes: float64(3), object(1)
memory usage: 845.4 KB


In [33]:
output_filename = f"arxiv_{search_term}_search_term_authors_info"
authors_info.to_parquet(f"{output_filename}.parquet")

In [34]:
authors_info.to_csvv(f"{output_filename}.tsv", sep = '\t')

AttributeError: 'DataFrame' object has no attribute 'to_csvv'

## 3. Co-authorship network

In [ ]:
N = 1000
author_subset = author_pair_df.sample(N)

rn = nx.from_pandas_edgelist(author_pair_df, source = "Author1", target = "Author1")
pprint(nx.info(rn).strip())

In [ ]:
# Source: https://plotly.com/python/v3/3d-network-graph/
# Numbers of Nodes
N = rn.number_of_nodes()

# List of Edge
L = rn.number_of_edges()

# Graph objects
# Edges Names
Edges_name = [e for e in rn.edges()]

# Mapping all Nodes into Numbers
Edges = nx.convert_node_labels_to_integers(rn)
Edges = [e for e in Edges.edges()]

# Graph
G = ig.Graph(Edges, directed = False)

# Geolocalization
# 3D Localization
layt = G.layout('kk',dim = 3)

# Given X,y,z Position
Xn = [layt[k][0] for k in range(N)] # x-coordinates of nodes
Yn = [layt[k][1] for k in range(N)] # y-coordinates
Zn = [layt[k][2] for k in range(N)] # z-coordinates

Xe = []
Ye = []
Ze = []

#Grouping Coordinates
for e in Edges:
    Xe += [layt[e[0]][0],layt[e[1]][0], None] # x-coordinates of edge ends
    Ye += [layt[e[0]][1],layt[e[1]][1], None]
    Ze += [layt[e[0]][2],layt[e[1]][2], None]
    
# Nodes Name 
labels = []
group = []

for i in range(len(Edges_name)):
    value = Edges_name[i][0]
    labels.append(value)
    
for i in range(len(Edges)):
    value = Edges[i][0]
    group.append(value)
    
group = []
group.extend(np.repeat(1,2000))
group.extend(np.repeat(2,2000))
group.extend(np.repeat(3,3000))
group.extend(np.repeat(4,1000))
group.extend(np.repeat(5,2000))

trace1 = go.Scatter3d(x = Xe, y = Ye, z = Ze, mode = 'lines',
                      line = dict(color = 'rgb(125,125,125)', width = 1), hoverinfo = 'none')

trace2 = go.Scatter3d(x = Xn,y = Yn, z = Zn, mode = 'markers', name = 'NLP Researcher',
                      marker = dict(symbol = 'circle', size = 4, color = group, colorscale = 'Viridis', 
                                    line = dict(color = 'rgb(50,50,50)', width = 0.5)),
                      text = labels, hoverinfo = 'text')

axis = dict(showbackground = False, showline = False, zeroline = False,
            showgrid = False, showticklabels = False, title = '')

In [ ]:
layout = go.Layout(
         title = f"Arxiv-с {search_term} түлхүүр үгийн дагуу хайлтаас илэрсэн өгүүллүүдийн хувьд тооцсон судлаачдын танилын сүлжээ",
         autosize = True,
         width = 1000,
         height = 1000,
         showlegend = False,
         scene = dict(
             xaxis = dict(axis),
             yaxis = dict(axis),
             zaxis = dict(axis),
        ),
        margin = dict(t = 100),
        hovermode = 'closest',
        annotations = [
            dict(
                showarrow = False,
                text = "Гүйцэтгэсэн Э.Тэмүүжин (20B1NUM1970) Хэрэглээний математик 4-р түвшин",
                xref = 'paper',
                yref = 'paper',
                x = 0,
                y = 0.1,
                xanchor = 'left',
                yanchor = 'bottom',
                font = dict(size = 14)
            )
        ])

data = [trace1, trace2]
fig = go.Figure(data = data, layout = layout)
fig.show();